# 07 - Ablation Comparison

Loads one or more `results/ablations/<run-id>/cells.jsonl` files and produces:

- A tidy DataFrame with one row per `(model, calibration, conformal, seed)` cell.
- A paper-ready aggregated table grouped by `(model, calibration, conformal)`.
- Heatmaps of coverage and prediction-set size.
- Box plots of ECE / accuracy across seeds for each variant.
- (If multiple runs are loaded) a delta table to compare two runs head-to-head.

Generate a run with:

```bash
uv run python scripts/run_ablation.py --seeds 42 123 456
```

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / "src"))

ABLATIONS_DIR = Path.cwd().parent / "results" / "ablations"
print(f"Ablations folder: {ABLATIONS_DIR}")
if ABLATIONS_DIR.exists():
    for d in sorted(ABLATIONS_DIR.iterdir()):
        if d.is_dir():
            print(f"  {d.name}")
else:
    print("  (no runs yet -- run scripts/run_ablation.py first)")

## 1. Load runs into a DataFrame

By default load the most recent run. Set `RUN_IDS` to a list to compare
multiple runs side by side.

In [ ]:
RUN_IDS: list[str] | None = None  # None = use the most recent run

def load_run(run_id: str) -> pd.DataFrame:
    folder = ABLATIONS_DIR / run_id
    rows = []
    with (folder / "cells.jsonl").open() as f:
        for line in f:
            obj = json.loads(line)
            row = {
                "run_id": run_id,
                "seed": obj["seed"],
                "model": obj["model"],
                "calibration": obj["calibration"],
                "conformal": obj["conformal"],
                "error": obj.get("error"),
            }
            row.update(obj.get("metrics", {}))
            row.update({f"time_{k}": v for k, v in obj.get("timing", {}).items()})
            rows.append(row)
    return pd.DataFrame(rows)

if RUN_IDS is None:
    runs = sorted([d.name for d in ABLATIONS_DIR.iterdir() if d.is_dir()])
    if not runs:
        raise RuntimeError(
            "No ablation runs found. Run `python scripts/run_ablation.py` first."
        )
    RUN_IDS = [runs[-1]]

df = pd.concat([load_run(r) for r in RUN_IDS], ignore_index=True)
df_ok = df[df["error"].isna()].copy()
print(f"Loaded runs: {RUN_IDS}")
print(f"Cells: {len(df)} ({len(df_ok)} non-error)")
df.head()

## 2. Paper-ready aggregated table

Mean ± std across seeds, grouped by `(model, calibration, conformal)`.

In [ ]:
metric_cols = ["accuracy", "ece", "mce", "brier"]
for col in ("coverage", "avg_set_size", "fraction_singleton", "ess_ratio"):
    if col in df_ok.columns:
        metric_cols.append(col)

agg = (
    df_ok
    .groupby(["model", "calibration", "conformal"], dropna=False)[metric_cols]
    .agg(["mean", "std", "count"])
)
agg.round(3)

## 3. Coverage heatmap by (model, conformal)

Average over seeds and calibration variants. Target = `1 - alpha` from the run
config (default 0.9).

In [ ]:
if "coverage" in df_ok.columns:
    pivot = (
        df_ok
        .groupby(["model", "conformal"])["coverage"]
        .mean()
        .unstack("conformal")
    )
    fig, ax = plt.subplots(figsize=(7, 3.5))
    sns.heatmap(
        pivot,
        annot=True, fmt=".2f", vmin=0.0, vmax=1.0, cmap="viridis",
        cbar_kws={"label": "Empirical coverage"}, ax=ax,
    )
    ax.set_title("Empirical conformal coverage (averaged over calibrations and seeds)")
    fig.tight_layout()
    plt.show()
else:
    print("No coverage column -- run did not include any conformal variant.")

## 4. ECE distribution by (model, calibration)

Box plot collapses across conformal variants and seeds.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(
    data=df_ok, x="model", y="ece", hue="calibration", ax=ax,
)
ax.set_title("ECE by (model, calibration) -- lower is better")
ax.axhline(0.05, ls="--", color="grey", alpha=0.6, label="target ECE 0.05")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## 5. Set-size vs coverage scatter

Useful for picking a Pareto-optimal `(calibration, conformal)` configuration:
we want high coverage (near 1) at low set size (near 1).

In [ ]:
if {"coverage", "avg_set_size"}.issubset(df_ok.columns):
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.scatterplot(
        data=df_ok.dropna(subset=["coverage", "avg_set_size"]),
        x="avg_set_size",
        y="coverage",
        hue="model",
        style="conformal",
        s=80,
        ax=ax,
    )
    ax.axhline(0.9, ls="--", color="grey", alpha=0.5, label="target 0.9")
    ax.set_xlabel("Average prediction set size (lower is more decisive)")
    ax.set_ylabel("Empirical coverage (higher is safer)")
    ax.set_title("Coverage vs set size -- Pareto front")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    plt.show()

## 6. Compare two runs (delta)

Skipped when only one run is loaded. Set `RUN_IDS = ["<run-a>", "<run-b>"]`
in cell 1 to enable.

In [ ]:
if df_ok["run_id"].nunique() < 2:
    print("Skipping run-vs-run delta (only one run loaded).")
else:
    delta_metrics = [c for c in ("ece", "coverage", "avg_set_size") if c in df_ok.columns]
    pivot = (
        df_ok
        .groupby(["run_id", "model", "calibration", "conformal"], dropna=False)[delta_metrics]
        .mean()
        .unstack("run_id")
    )
    print(pivot.round(3))
    if pivot.columns.nlevels == 2 and pivot.shape[1] >= 2:
        a_id, b_id = pivot.columns.get_level_values(1).unique()[:2]
        delta = (pivot.xs(b_id, level=1, axis=1) - pivot.xs(a_id, level=1, axis=1))
        delta.columns = [f"delta_{c}" for c in delta.columns]
        print(f"\nDelta = {b_id} minus {a_id}:")
        print(delta.round(3))

## 7. Run config + reproducibility

Every run captures git hash, library versions, timestamp, and the variant lists
actually requested. Useful when comparing runs from different commits.

In [ ]:
for run_id in RUN_IDS:
    cfg_path = ABLATIONS_DIR / run_id / "config.json"
    print(f"# {run_id}")
    print(json.dumps(json.loads(cfg_path.read_text()), indent=2))
    print()